## Финальный проект (RAG для диалоговых систем)

В этом проекте мы создадим ассистента, который сможет отвечать на любые вопросы про жизнь известных личностей. Для этого мы реализуем поддержку диалога в RAG, а также к семантическому поиску по базе знаний мы добавим поиск информации в интернете. Поддержка диалога означает, что пользователь сможет уточнять любую информацию по предыдущему вопросу без необходимости задавать весь вопрос целиком.

### База знаний

База знаний состоит из первых абзацев русскоязычных статей из википедии про различных людей.

In [1]:
with open('data/ru_wiki_person.txt', 'r') as f:
    articles = f.read().split('\n\n')

len(articles)

269086

In [ ]:
articles[:5]

### Задание

В этом задании у вас будет гораздо больше свободы в реализации системы и не будет подсказок о том, как имплементировать те или иные компоненты. Вам предстоит самостоятельно организовать логику работы системы от начала до конца. Однако мы все же наметим план, которого стоит придерживаться:

1. Собрать векторную базу данных.
2. Написать движок для поиска текстов по базе данных.
3. Добавить функцию поиска текстов в интернете.
4. Добавить поддержку диалогового режима.
5. Составить из полученных компонент RAG и протестировать его работу.

Приступим! Ниже будет набор заданий с минимальной реализацией компонент, необходимых для RAG. Предполагается, для построения итоговой системы вы усложните данные компоненты по своему усмотрению.

__Задание 1.__ Создайте базу данных из __первых 100__ текстов в датасете. Вам предлагается использовать [ChromaDB](https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/) из langchain. Она работает аналогично Qdrant, но, помимо всего прочего, ее проще сохранять на диск после создания. Это очень важно сделать, чтобы не считать эмбеддинги каждый раз заново.

Cохраните базу данных на диск с названием `chroma_db`. Никак не обрабатывайте тексты дополнительно (при построении RAG, вам, конечно, нужно будет резать тексты на куски). В грейдер сдайте zip архив с полученной базой данных ChromaDB. Мы будем загружать ее таким образом.
```
import zipfile

with zipfile.ZipFile('chroma_db.zip', 'r') as zip_ref:
    zip_ref.extractall('./')

db = Chroma(persist_directory="chroma_db", embedding_function=embedding_model)
```

Имя коллекции `collection_name` оставляйте в значении по умолчанию, иначе грейдер сломается. Как и раньше, в качестве модели эмбеддингов используйте `intfloat/multilingual-e5-large` из huggingface.

In [ ]:
# ваш код здесь

### Retrieval Augmented Generation

Теперь можно собрать полную векторную базу данных и дописать вторую часть RAG – генерацию ответа. В качестве генеративной модели выберите `hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4` из `huggingface`. Это квантизованная версия Llama 3.1, которая отлично генерирует текст как на английском, так и на русском языке. Заметьте, что AWQ работает не на всех видеокартах. Например, такая квантизация не поддерживается на V100. Загрузить модель можно таким образом.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4")
model = AutoModelForCausalLM.from_pretrained("hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4")

__Задание 2.__ С помощью RAG сгенерируйте ответы к вопросам из файла `questions.txt`. Постарайтесь подобрать основной промпт таким образом, чтобы ответ был коротким и четким. Результат генерации сохраните в файл `answers.json` в виде списка ответов. 

```
import json

with open('answers.json', 'w', encoding='utf8') as f:
    json.dump(generated_answers, f, ensure_ascii=False)
```

In [ ]:
# ваш код здесь

### Поиск в интернете

Поиск в интернете можно использовать в том случае, если в базе знаний не нашлось достаточно подходящих текстов. Например, в Википедии ничего не написано про Александра Шабалина. Так что если вы спросите, кто является автором курса по NLP в karpov.courses, то без поиска в интернете, модель не сможет дать правильный ответ.

__Заданиe 3.__
Напишите функцию `internet_search`, которая принимает на вход текстовый запрос и аргумент `k` и возвращает набор из `k` текстов, найденных в интернете по полученному запросу. В качестве браузера проще всего использовать [`DuckDuckGO`](https://duckduckgo.com/) и специализированную [библиотеку](https://pypi.org/project/duckduckgo-search/) для него. Также скорее всего вам пригодятся библиотеки [`requests`](https://requests.readthedocs.io/en/latest/) и [`BeautifulSoup`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).

При встраивании этой компоненты в RAG подумайте о том, как понять, что релевантных текстов не оказалось в базе данных, а так же о том, какие тексты (куски?) и в каком количестве надо добавлять в контекст модели.

In [ ]:
# ваш код здесь

### Поддержка диалогов

Когда модель умеет отвечать на один поставленный вопрос - это хорошо. Но когда она умеет отвечать на уточняющие вопросы, учитывая историю общения – это еще лучше.

__Пример:__    
    – _Пользователь_: Кто был самым высоким человеком?   
    – _Ассистент_: Роберт Уодлоу.   
    – _Пользователь_: Какой у него был рост?   
    – _Ассистент_: 272 сантиметров.   

__Задание 4.__ Добавьте поддержку диалога в вашу систему RAG. С данной модификацией сгенерируйте ответы на вопросы
из файла `dialog_questions.txt` и запишите результат в файл `dialog_answers.json` в виде списка из пар ответов: ответ на первый вопрос и ответ на второй вопрос.  Если нужных документов нет в базе данных, используйте поиск в интернете.

_Подсказка:_ Для того, чтобы по новому вопросу можно было достать релевантные тексты из базы данных, вопрос нужно переформулировать, добавив нужную информацию из предыдущих сообщений пользователя. Поэтому при получении нового вопроса можно сделать запрос в LLM для уточнения запроса пользователя с учетом всей истории сообщений, а после этого искать релевантные тексты по уточненному запросу.

In [ ]:
# ваш код здесь

### Резюме

Ура! Теперь у вас есть ассистент, который с легкостью может заменить гугл. Если вы добавите к нему пользовательский интерфейс, то получите самый удобный способ поиска ответов на вопросы о людях. Это решение можно развивать и дальше, как улучшая имеющиеся компоненты, так и добавляя новые. Однако в рамках финального проекта мы остановимся на том, что есть.

Мы благодарим вас за прохождение данного курса и очень надеемся, что вы получили те знания, которые хотели, или даже больше. По крайней мере, теперь вы можете смело называть себя NLP-инженером :)